## 2. Задание

### 2.3.1 Импорты, seed и устройство

Импортируем необходимые библиотеки, фиксируем seed и определяем устройство.

In [66]:
import json
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms

# Seed 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Устройство
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    '''elif torch.backends.mps.is_available():
    # для Mac 
    DEVICE = torch.device("mps")
    torch.backends.mps.benchmark = False
    torch.backends.mps.deterministic = True'''
else:
    DEVICE = torch.device("cpu")
    torch.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__}  |  torchvision {torchvision.__version__}")
print(f"Device : {DEVICE}")
print(f"Seed   : {SEED}")

PyTorch 2.10.0  |  torchvision 0.25.0
Device : cpu
Seed   : 42


### 2.3.2 Данные и DataLoader

Загружаем EMNIST balanced, делаем воспроизводимое разбиение train/val (80/20) и создаём DataLoader для train/val/test и sanity-check.

In [67]:
# Параметры
BATCH_SIZE = 256 if DEVICE.type in ('cuda', 'mps') else 128
VAL_RATIO  = 0.2        # 20 % от train --> val
NUM_CLASSES = 47         # EMNIST balanced: 47 классов (10 цифр + 37 букв)

# Трансформации
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# Загрузка EMNIST Balanced
full_train = torchvision.datasets.EMNIST(root="./data", train=True,
                                         download=False, transform=transform,
                                         split="balanced")
test_ds    = torchvision.datasets.EMNIST(root="./data", train=False,
                                         download=False, transform=transform,
                                         split="balanced")

# Разбиение train / val (80 / 20) с фиксированным seed
n_val   = int(len(full_train) * VAL_RATIO)
n_train = len(full_train) - n_val
train_ds, val_ds = random_split(full_train, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))

# DataLoader'ы
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train : {len(train_ds)}")
print(f"Val   : {len(val_ds)}")
print(f"Test  : {len(test_ds)}")

# Sanity-check
x_batch, y_batch = next(iter(train_loader))
print(f"\nx.shape = {x_batch.shape}  |  y.shape = {y_batch.shape}")
print(f"x min/max = {x_batch.min():.2f} | {x_batch.max():.2f}")
print(f"Число классов: {len(full_train.classes)}")

Train : 90240
Val   : 22560
Test  : 18800

x.shape = torch.Size([128, 1, 28, 28])  |  y.shape = torch.Size([128])
x min/max = -1.00 | 1.00
Число классов: 47


### 2.3.3 Модель MLP и цикл обучения

MLP как `nn.Module`, функции `train_one_epoch()` и `evaluate()` с корректным переключением `model.train()` / `model.eval()` и `torch.no_grad()`.

In [68]:
class MLP(nn.Module):
    def __init__(
            self, input_dim=784, 
            num_classes=47,
            hidden_sizes=(512, 256), 
            dropout_p=0.0, 
            use_batchnorm=False):
        super().__init__()
        layers = [nn.Flatten()]
        prev = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            prev = h
        layers.append(nn.Linear(prev, num_classes))   # выход (logits)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# Цикл обучения за 1 эпоху
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train() # train-режим
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking = True), y.to(device, non_blocking = True)

        optimizer.zero_grad(set_to_none = True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return running_loss / total, correct / total


# Оценка (val / test)
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval() # eval-моды
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total


# Полный цикл обучения с EarlyStopping
def fit(model,
        train_loader, 
        val_loader, 
        criterion, 
        optimizer,
        device, 
        max_epochs=20, 
        early_stopping=None,
        verbose = True):
    
    history = {"train_loss": [], "val_loss": [],
               "train_acc": [], "val_acc": []}
    
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                          criterion, optimizer, device)
        vl_loss, vl_acc = evaluate(model, val_loader, 
                                   criterion, device)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        tag = ""
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            tag = " ★"
        else:
            epochs_no_improve += 1

        if verbose:
            print(f"epoch {epoch:>2d}/{max_epochs} | "
                f"train_loss={tr_loss:.4f} , train_acc={tr_acc:.4f} | "
                f"val_loss={vl_loss:.4f} , val_acc={vl_acc:.4f}{tag}")

        # EarlyStopping
        if early_stopping is not None:
            should_stop = early_stopping.step(vl_acc, model)
            if should_stop:
                if verbose:
                    print(
                        f"EarlyStopping: остановка на эпохе {epoch}. "
                        f"Лучший val_acc={early_stopping.best_score:.4f}"
                    )
                early_stopping.restore_best(model)
                break

    return history, best_state, best_val_acc


class EarlyStopping:
    def __init__(self,
                 patience=5,
                 min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.best_state = None
        self.counter = 0
    def step(self, score: float, model: nn.Module) -> bool:
        # score: чем больше, тем лучше (например, val_acc).
        # Возвращает True, если нужно остановиться.
        if self.best_score is None:
            self.best_score = score
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False

        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
            return False

        self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

        

## 3. Эксперименты

### 3.1 Часть A (S08): регуляризация и переобучение (E1–E4)

Проводим 4 эксперимента для сравнения влияния Dropout, BatchNorm и EarlyStopping.
Во всех экспериментах E1–E3 используется одинаковый optimizer (Adam, lr=1e-3), batch size, число эпох и разбиение данных.
    E1 (base) - MLP побольше, без Dropout, без BatchNorm 
    E2 (Dropout) - как E1 + Dropout(p=0.3) 
    E3 (BatchNorm) - как E1 + BatchNorm 
    E4 (EarlyStopping) - лучший из E2/E3 + EarlyStopping (patience=5)

#### E1 — MLP побольше (без Dropout, без BatchNorm)

MLP с двумя скрытыми слоями [512, 256], активация ReLU. Обучаем 20 эпох с Adam (lr=1e-3).

In [69]:
print("E1: MLP (no Dropout, no BatchNorm)")

torch.manual_seed(SEED)
np.random.seed(SEED)

model_e1 = MLP(hidden_sizes=[512, 256], 
               dropout_p=0.0, 
               use_batchnorm=False).to(DEVICE)

criterion = nn.CrossEntropyLoss() #для всех
optimizer_e1 = optim.Adam(model_e1.parameters(), lr=1e-3)

history_e1, state_e1, best_val_e1 = fit(
    model_e1, train_loader,
    val_loader, criterion, 
    optimizer_e1, DEVICE, 
    max_epochs=20, early_stopping=None
    )

print(f"\nE1 best val_accuracy = {best_val_e1:.4f}")

E1: MLP (no Dropout, no BatchNorm)
epoch  1/20 | train_loss=1.1270 , train_acc=0.6675 | val_loss=0.7397 , val_acc=0.7688 ★
epoch  2/20 | train_loss=0.6255 , train_acc=0.7977 | val_loss=0.6130 , val_acc=0.7987 ★
epoch  3/20 | train_loss=0.5209 , train_acc=0.8239 | val_loss=0.5386 , val_acc=0.8208 ★
epoch  4/20 | train_loss=0.4590 , train_acc=0.8410 | val_loss=0.5253 , val_acc=0.8268 ★
epoch  5/20 | train_loss=0.4199 , train_acc=0.8519 | val_loss=0.4950 , val_acc=0.8348 ★
epoch  6/20 | train_loss=0.3898 , train_acc=0.8604 | val_loss=0.4871 , val_acc=0.8381 ★
epoch  7/20 | train_loss=0.3629 , train_acc=0.8666 | val_loss=0.4896 , val_acc=0.8402 ★
epoch  8/20 | train_loss=0.3412 , train_acc=0.8729 | val_loss=0.4814 , val_acc=0.8436 ★
epoch  9/20 | train_loss=0.3202 , train_acc=0.8794 | val_loss=0.5003 , val_acc=0.8427
epoch 10/20 | train_loss=0.3018 , train_acc=0.8837 | val_loss=0.4975 , val_acc=0.8415
epoch 11/20 | train_loss=0.2858 , train_acc=0.8890 | val_loss=0.5048 , val_acc=0.8452 ★
e

#### E2 — Dropout (p=0.3)

Та же MLP [512, 256], но после каждого ReLU добавлен Dropout(p=0.3). Ожидаем уменьшение переобучения.

In [70]:
print("E2: Dropout (p=0.3)")

torch.manual_seed(SEED)
np.random.seed(SEED)

model_e2 = MLP(hidden_sizes=[512, 256], 
               dropout_p=0.3, 
               use_batchnorm=False).to(DEVICE)

optimizer_e2 = optim.Adam(model_e2.parameters(), lr=1e-3)

history_e2, state_e2, best_val_e2 = fit(
    model_e2, train_loader, 
    val_loader, criterion, 
    optimizer_e2, DEVICE, 
    max_epochs=20, early_stopping=None
)
print(f"\nE2 best val_accuracy = {best_val_e2:.4f}")

E2: Dropout (p=0.3)
epoch  1/20 | train_loss=1.3830 , train_acc=0.5944 | val_loss=0.7812 , val_acc=0.7609 ★
epoch  2/20 | train_loss=0.8521 , train_acc=0.7333 | val_loss=0.6423 , val_acc=0.7943 ★
epoch  3/20 | train_loss=0.7466 , train_acc=0.7607 | val_loss=0.5896 , val_acc=0.8069 ★
epoch  4/20 | train_loss=0.6914 , train_acc=0.7758 | val_loss=0.5578 , val_acc=0.8171 ★
epoch  5/20 | train_loss=0.6552 , train_acc=0.7848 | val_loss=0.5223 , val_acc=0.8273 ★
epoch  6/20 | train_loss=0.6278 , train_acc=0.7902 | val_loss=0.5080 , val_acc=0.8321 ★
epoch  7/20 | train_loss=0.6086 , train_acc=0.7968 | val_loss=0.4999 , val_acc=0.8314
epoch  8/20 | train_loss=0.5951 , train_acc=0.7992 | val_loss=0.5023 , val_acc=0.8315
epoch  9/20 | train_loss=0.5834 , train_acc=0.8025 | val_loss=0.4848 , val_acc=0.8331 ★
epoch 10/20 | train_loss=0.5697 , train_acc=0.8054 | val_loss=0.4727 , val_acc=0.8430 ★
epoch 11/20 | train_loss=0.5631 , train_acc=0.8080 | val_loss=0.4835 , val_acc=0.8417
epoch 12/20 | trai

#### E3 — BatchNorm

Та же MLP [512, 256], но есть BatchNorm1d (перед ReLU). Без Dropout.

In [71]:
print("E3: BatchNorm")

torch.manual_seed(SEED)
np.random.seed(SEED)

model_e3 = MLP(hidden_sizes=[512, 256], 
               dropout_p=0.0, 
               use_batchnorm=True).to(DEVICE)

optimizer_e3 = optim.Adam(model_e3.parameters(), lr=1e-3)

history_e3, state_e3, best_val_e3 = fit(
    model_e3, train_loader, 
    val_loader, criterion, 
    optimizer_e3, DEVICE, 
    max_epochs=20, early_stopping=None
)
print(f"\nE3 best val_accuracy = {best_val_e3:.4f}")

E3: BatchNorm
epoch  1/20 | train_loss=0.8501 , train_acc=0.7496 | val_loss=0.5626 , val_acc=0.8191 ★
epoch  2/20 | train_loss=0.4874 , train_acc=0.8355 | val_loss=0.5039 , val_acc=0.8289 ★
epoch  3/20 | train_loss=0.4109 , train_acc=0.8563 | val_loss=0.4679 , val_acc=0.8427 ★
epoch  4/20 | train_loss=0.3631 , train_acc=0.8693 | val_loss=0.4548 , val_acc=0.8442 ★
epoch  5/20 | train_loss=0.3263 , train_acc=0.8801 | val_loss=0.4506 , val_acc=0.8486 ★
epoch  6/20 | train_loss=0.2991 , train_acc=0.8869 | val_loss=0.4593 , val_acc=0.8487 ★
epoch  7/20 | train_loss=0.2753 , train_acc=0.8955 | val_loss=0.4667 , val_acc=0.8483
epoch  8/20 | train_loss=0.2539 , train_acc=0.9019 | val_loss=0.4673 , val_acc=0.8496 ★
epoch  9/20 | train_loss=0.2366 , train_acc=0.9069 | val_loss=0.4712 , val_acc=0.8502 ★
epoch 10/20 | train_loss=0.2232 , train_acc=0.9117 | val_loss=0.4700 , val_acc=0.8510 ★
epoch 11/20 | train_loss=0.2086 , train_acc=0.9161 | val_loss=0.4945 , val_acc=0.8500
epoch 12/20 | train_lo

#### E4 — EarlyStopping (лучший из E2/E3 + patience=5)

Выбираем лучший конфиг из E2 и E3 по val_accuracy, затем обучаем его заново с EarlyStopping (patience=5). Именно E4 сохраняется как лучшая модель.

In [72]:
if best_val_e2 >= best_val_e3:
    best_source = "E2 (Dropout)"
    e4_dropout = 0.3
    e4_batchnorm = False
else:
    best_source = "E3 (BatchNorm)"
    e4_dropout = 0.0
    e4_batchnorm = True

print(f"E4: EarlyStopping — на основе {best_source}")
print(f"    (E2 val_acc={best_val_e2:.4f}, E3 val_acc={best_val_e3:.4f})\n")

torch.manual_seed(SEED)
np.random.seed(SEED)

model_e4 = MLP(hidden_sizes=[512, 256],
               dropout_p=e4_dropout, 
               use_batchnorm=e4_batchnorm).to(DEVICE)

optimizer_e4 = optim.Adam(model_e4.parameters(), lr=1e-3)
es = EarlyStopping(patience=5, min_delta=0.0005)

history_e4, state_e4, best_val_e4 = fit(
    model_e4, train_loader, 
    val_loader, criterion, 
    optimizer_e4, DEVICE, 
    max_epochs=30, early_stopping=es   # больше эпох с ранней остановкой
)
epochs_e4 = len(history_e4["train_loss"])
print(f"\nE4 best val_accuracy = {best_val_e4:.4f}  (остановка на эпохе {epochs_e4})")

E4: EarlyStopping — на основе E3 (BatchNorm)
    (E2 val_acc=0.8473, E3 val_acc=0.8517)

epoch  1/30 | train_loss=0.8501 , train_acc=0.7496 | val_loss=0.5626 , val_acc=0.8191 ★
epoch  2/30 | train_loss=0.4874 , train_acc=0.8355 | val_loss=0.5039 , val_acc=0.8289 ★
epoch  3/30 | train_loss=0.4109 , train_acc=0.8563 | val_loss=0.4679 , val_acc=0.8427 ★
epoch  4/30 | train_loss=0.3631 , train_acc=0.8693 | val_loss=0.4548 , val_acc=0.8442 ★
epoch  5/30 | train_loss=0.3263 , train_acc=0.8801 | val_loss=0.4506 , val_acc=0.8486 ★
epoch  6/30 | train_loss=0.2991 , train_acc=0.8869 | val_loss=0.4593 , val_acc=0.8487 ★
epoch  7/30 | train_loss=0.2753 , train_acc=0.8955 | val_loss=0.4667 , val_acc=0.8483
epoch  8/30 | train_loss=0.2539 , train_acc=0.9019 | val_loss=0.4673 , val_acc=0.8496 ★
epoch  9/30 | train_loss=0.2366 , train_acc=0.9069 | val_loss=0.4712 , val_acc=0.8502 ★
epoch 10/30 | train_loss=0.2232 , train_acc=0.9117 | val_loss=0.4700 , val_acc=0.8510 ★
epoch 11/30 | train_loss=0.2086 ,

#### Сравнение E1–E4 и графики для 3.1

Таблица val_accuracy и кривые train/val loss для всех экспериментов части A.
Сохраняем curves_best.png — кривые лучшего прогона E4.

In [73]:
print("Сводка Part A:")
print(f"  E1 (base)      val_acc = {best_val_e1:.4f}")
print(f"  E2 (Dropout)   val_acc = {best_val_e2:.4f}")
print(f"  E3 (BatchNorm) val_acc = {best_val_e3:.4f}")
print(f"  E4 (EarlyStop) val_acc = {best_val_e4:.4f}  (epochs={epochs_e4})")

# Общий plot для первой части 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, h in [("E1-base", history_e1), ("E2-Dropout", history_e2),
                ("E3-BN", history_e3), ("E4-EarlyStop", history_e4)]:
    axes[0].plot(h["train_loss"], label=f"{name} train", linestyle="--", alpha=0.6)
    axes[0].plot(h["val_loss"],   label=f"{name} val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Train / Val Loss — E1‑E4")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

for name, h in [("E1-base", history_e1), ("E2-Dropout", history_e2),
                ("E3-BN", history_e3), ("E4-EarlyStop", history_e4)]:
    axes[1].plot(h["val_acc"], label=name)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val Accuracy")
axes[1].set_title("Val Accuracy — E1‑E4")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("artifacts/figures/curves_partA.png", dpi=150)
plt.show()

# curves_best.png
fig2, ax2 = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, epochs_e4 + 1)

ax2[0].plot(epochs_range, history_e4["train_loss"], label="train loss")
ax2[0].plot(epochs_range, history_e4["val_loss"],   label="val loss")
ax2[0].set_xlabel("Epoch"); ax2[0].set_ylabel("Loss")
ax2[0].set_title("E4 (best) — Loss"); ax2[0].legend(); ax2[0].grid(True, alpha=0.3)

ax2[1].plot(epochs_range, history_e4["train_acc"], label="train acc")
ax2[1].plot(epochs_range, history_e4["val_acc"],   label="val acc")
ax2[1].set_xlabel("Epoch"); ax2[1].set_ylabel("Accuracy")
ax2[1].set_title("E4 (best) — Accuracy"); ax2[1].legend(); ax2[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("artifacts/figures/curves_best.png", dpi=150)
plt.show()
print("Сохранено: artifacts/figures/curves_best.png")

Сводка Part A:
  E1 (base)      val_acc = 0.8452
  E2 (Dropout)   val_acc = 0.8473
  E3 (BatchNorm) val_acc = 0.8517
  E4 (EarlyStop) val_acc = 0.8510  (epochs=15)
Сохранено: artifacts/figures/curves_best.png


/var/folders/2h/tj3pf8h10qx0cb_6ly1sbtv80000gn/T/ipykernel_9746/3902505072.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/2h/tj3pf8h10qx0cb_6ly1sbtv80000gn/T/ipykernel_9746/3902505072.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Из графика видно, что соответственно лучшая модель это E4 (которая была основана на E3). Худшие E2 и E1.

### 3.2 Часть B (S09): LR, оптимизаторы, weight decay (O1–O3)

Диагностика плохого learning rate и сравнение Adam vs SGD+momentum.
Архитектура фиксирована — та же, что в E4.
O1 - Adam, lr=0.1 (слишком большой шаг) — 8 эпох 
O2 - Adam, lr=1e-5 (слишком маленький шаг) — 8 эпох 
O3 - SGD+momentum(0.9) + weight_decay=1e-4, lr=1e-2 — 15 эпох 

#### O1 — LR слишком большой (Adam, lr=0.1)

Ожидаем нестабильный loss и низкую accuracy.

In [74]:
print("O1: Adam, lr=0.1 (too high)")

torch.manual_seed(SEED)
np.random.seed(SEED)

model_o1 = MLP(hidden_sizes=[512, 256],
               dropout_p=e4_dropout, 
               use_batchnorm=e4_batchnorm).to(DEVICE)

optimizer_o1 = optim.Adam(model_o1.parameters(), lr=0.1)

history_o1, _, best_val_o1 = fit(
    model_o1, train_loader, 
    val_loader, criterion, 
    optimizer_o1, DEVICE, 
    max_epochs=8, early_stopping=None
)
print(f"\nO1 best val_accuracy = {best_val_o1:.4f}")

O1: Adam, lr=0.1 (too high)
epoch  1/8 | train_loss=1.1217 , train_acc=0.6673 | val_loss=0.8333 , val_acc=0.7463 ★
epoch  2/8 | train_loss=0.7548 , train_acc=0.7594 | val_loss=0.7090 , val_acc=0.7762 ★
epoch  3/8 | train_loss=0.6772 , train_acc=0.7797 | val_loss=0.6832 , val_acc=0.7858 ★
epoch  4/8 | train_loss=0.6309 , train_acc=0.7924 | val_loss=0.6566 , val_acc=0.7955 ★
epoch  5/8 | train_loss=0.5999 , train_acc=0.8013 | val_loss=0.6407 , val_acc=0.7992 ★
epoch  6/8 | train_loss=0.5788 , train_acc=0.8081 | val_loss=0.6725 , val_acc=0.7917
epoch  7/8 | train_loss=0.5542 , train_acc=0.8142 | val_loss=0.6194 , val_acc=0.8012 ★
epoch  8/8 | train_loss=0.5400 , train_acc=0.8193 | val_loss=0.6106 , val_acc=0.8102 ★

O1 best val_accuracy = 0.8102


#### O2 — LR слишком маленький (Adam, lr=1e-5)

Обучение почти не двигается: loss снижается крайне медленно, accuracy остаётся низкой.

In [75]:
print("O2: Adam, lr=1e-5 (too low)")

torch.manual_seed(SEED)
np.random.seed(SEED)

model_o2 = MLP(hidden_sizes=[512, 256],
               dropout_p=e4_dropout, 
               use_batchnorm=e4_batchnorm).to(DEVICE)

optimizer_o2 = optim.Adam(model_o2.parameters(), lr=1e-5)

history_o2, _, best_val_o2 = fit(
    model_o2, train_loader, 
    val_loader, criterion, 
    optimizer_o2, DEVICE, 
    max_epochs=8, early_stopping=None
)
print(f"\nO2 best val_accuracy = {best_val_o2:.4f}")

O2: Adam, lr=1e-5 (too low)
epoch  1/8 | train_loss=3.1833 , train_acc=0.2914 | val_loss=2.6732 , val_acc=0.4690 ★
epoch  2/8 | train_loss=2.4086 , train_acc=0.5298 | val_loss=2.1703 , val_acc=0.5748 ★
epoch  3/8 | train_loss=2.0119 , train_acc=0.6025 | val_loss=1.8479 , val_acc=0.6252 ★
epoch  4/8 | train_loss=1.7432 , train_acc=0.6435 | val_loss=1.6227 , val_acc=0.6575 ★
epoch  5/8 | train_loss=1.5427 , train_acc=0.6725 | val_loss=1.4532 , val_acc=0.6826 ★
epoch  6/8 | train_loss=1.3869 , train_acc=0.6933 | val_loss=1.3236 , val_acc=0.7018 ★
epoch  7/8 | train_loss=1.2592 , train_acc=0.7131 | val_loss=1.2104 , val_acc=0.7191 ★
epoch  8/8 | train_loss=1.1550 , train_acc=0.7276 | val_loss=1.1167 , val_acc=0.7325 ★

O2 best val_accuracy = 0.7325


#### O3 — SGD+momentum + weight decay

SGD с momentum=0.9 и weight_decay=1e-4, lr=1e-2. Обучаем 15 эпох. Сравним с результатами Adam из E4.

In [76]:
print("O3: SGD + momentum=0.9, weight_decay=1e-4, lr=0.01")

torch.manual_seed(SEED)
np.random.seed(SEED)

model_o3 = MLP(hidden_sizes=[512, 256],
               dropout_p=e4_dropout, 
               use_batchnorm=e4_batchnorm).to(DEVICE)

optimizer_o3 = optim.SGD(model_o3.parameters(), lr=0.01,
                         momentum=0.9, weight_decay=1e-4)

history_o3, _, best_val_o3 = fit(
    model_o3, train_loader, 
    val_loader, criterion, 
    optimizer_o3, DEVICE, 
    max_epochs=15, early_stopping=None
)
print(f"\nO3 best val_accuracy = {best_val_o3:.4f}")

O3: SGD + momentum=0.9, weight_decay=1e-4, lr=0.01
epoch  1/15 | train_loss=0.9965 , train_acc=0.7200 | val_loss=0.6165 , val_acc=0.8059 ★
epoch  2/15 | train_loss=0.5364 , train_acc=0.8254 | val_loss=0.5228 , val_acc=0.8258 ★
epoch  3/15 | train_loss=0.4478 , train_acc=0.8479 | val_loss=0.4837 , val_acc=0.8393 ★
epoch  4/15 | train_loss=0.3961 , train_acc=0.8616 | val_loss=0.4724 , val_acc=0.8413 ★
epoch  5/15 | train_loss=0.3589 , train_acc=0.8728 | val_loss=0.4596 , val_acc=0.8443 ★
epoch  6/15 | train_loss=0.3300 , train_acc=0.8809 | val_loss=0.4535 , val_acc=0.8483 ★
epoch  7/15 | train_loss=0.3050 , train_acc=0.8897 | val_loss=0.4536 , val_acc=0.8494 ★
epoch  8/15 | train_loss=0.2839 , train_acc=0.8953 | val_loss=0.4548 , val_acc=0.8466
epoch  9/15 | train_loss=0.2673 , train_acc=0.9017 | val_loss=0.4572 , val_acc=0.8484
epoch 10/15 | train_loss=0.2522 , train_acc=0.9059 | val_loss=0.4613 , val_acc=0.8453
epoch 11/15 | train_loss=0.2402 , train_acc=0.9093 | val_loss=0.4587 , val_

#### Графики Part B — LR extremes и сравнение O1/O2/O3

Визуализируем поведение loss/accuracy при плохом LR (O1, O2) и SGD+momentum (O3).
Сохраняем curves_lr_extremes.png.

In [77]:
# curves_lr_extremes.png
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, h, ls in [("O1 lr=0.1", history_o1, "-"),
                     ("O2 lr=1e-5", history_o2, "-"),
                     ("O3 SGD+mom", history_o3, "--")]:
    axes[0].plot(h["val_loss"], label=name, linestyle=ls)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val Loss")
axes[0].set_title("Val Loss — O1 / O2 / O3")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for name, h, ls in [("O1 lr=0.1", history_o1, "-"),
                     ("O2 lr=1e-5", history_o2, "-"),
                     ("O3 SGD+mom", history_o3, "--")]:
    axes[1].plot(h["val_acc"], label=name, linestyle=ls)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Accuracy")
axes[1].set_title("Val Accuracy — O1 / O2 / O3")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("artifacts/figures/curves_lr_extremes.png", dpi=150)
plt.show()
print("Сохранено: artifacts/figures/curves_lr_extremes.png")

Сохранено: artifacts/figures/curves_lr_extremes.png


/var/folders/2h/tj3pf8h10qx0cb_6ly1sbtv80000gn/T/ipykernel_9746/1647277384.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Финальная оценка лучшей модели (E4) на test

Test-выборка используется один раз — только для финальной оценки лучшей модели (E4).

In [78]:
# Загружаем лучшие веса E4
model_best = MLP(hidden_sizes=[512, 256],
                 dropout_p=e4_dropout, 
                 use_batchnorm=e4_batchnorm).to(DEVICE)

model_best.load_state_dict(state_e4)

test_loss, test_acc = evaluate(model_best, 
                               test_loader, 
                               criterion, 
                               DEVICE)

print(f"test  loss = {test_loss:.4f}  |  accuracy = {test_acc:.4f}")

test  loss = 0.4961  |  accuracy = 0.8470


### Сохранение артефактов

Сохраняем все обязательные файлы:
- `artifacts/runs.csv` — таблица результатов E1–E4 и O1–O3
- `artifacts/best_model.pt` — state_dict лучшей модели (E4)
- `artifacts/best_config.json` — конфиг лучшей модели

In [79]:
# 1) runs.csv
def model_summary(dp, bn):
    parts = ["[512,256] ReLU"]
    if dp > 0:  parts.append(f"Dropout({dp})")
    if bn:      parts.append("BatchNorm")
    return " | ".join(parts)

rows = [
    {"experiment_id": "E1", "dataset": "EMNIST Balanced", "seed": SEED,
     "model_summary": model_summary(0.0, False),
     "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
     "epochs_trained": 20,
     "best_val_accuracy": round(best_val_e1, 4),
     "best_val_loss": round(min(history_e1["val_loss"]), 4)},

    {"experiment_id": "E2", "dataset": "EMNIST Balanced", "seed": SEED,
     "model_summary": model_summary(0.3, False),
     "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
     "epochs_trained": 20,
     "best_val_accuracy": round(best_val_e2, 4),
     "best_val_loss": round(min(history_e2["val_loss"]), 4)},

    {"experiment_id": "E3", "dataset": "EMNIST Balanced", "seed": SEED,
     "model_summary": model_summary(0.0, True),
     "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
     "epochs_trained": 20,
     "best_val_accuracy": round(best_val_e3, 4),
     "best_val_loss": round(min(history_e3["val_loss"]), 4)},

    {"experiment_id": "E4", "dataset": "EMNIST Balanced", "seed": SEED,
     "model_summary": model_summary(e4_dropout, e4_batchnorm),
     "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
     "epochs_trained": epochs_e4,
     "best_val_accuracy": round(best_val_e4, 4),
     "best_val_loss": round(min(history_e4["val_loss"]), 4)},

    {"experiment_id": "O1", "dataset": "EMNIST Balanced", "seed": SEED,
     "model_summary": model_summary(e4_dropout, e4_batchnorm),
     "optimizer": "Adam", "lr": 0.1, "momentum": 0, "weight_decay": 0,
     "epochs_trained": 8,
     "best_val_accuracy": round(best_val_o1, 4),
     "best_val_loss": round(min(history_o1["val_loss"]), 4)},

    {"experiment_id": "O2", "dataset": "EMNIST Balanced", "seed": SEED,
     "model_summary": model_summary(e4_dropout, e4_batchnorm),
     "optimizer": "Adam", "lr": 1e-5, "momentum": 0, "weight_decay": 0,
     "epochs_trained": 8,
     "best_val_accuracy": round(best_val_o2, 4),
     "best_val_loss": round(min(history_o2["val_loss"]), 4)},

    {"experiment_id": "O3", "dataset": "EMNIST Balanced", "seed": SEED,
     "model_summary": model_summary(e4_dropout, e4_batchnorm),
     "optimizer": "SGD", "lr": 0.01, "momentum": 0.9, "weight_decay": 1e-4,
     "epochs_trained": 15,
     "best_val_accuracy": round(best_val_o3, 4),
     "best_val_loss": round(min(history_o3["val_loss"]), 4)},
]

df_runs = pd.DataFrame(rows)
df_runs.to_csv("artifacts/runs.csv", index=False)
print("artifacts/runs.csv complete")
print(df_runs.to_string(index=False))

# 2) best_model.pt
torch.save(state_e4, "artifacts/best_model.pt")
print("\nartifacts/best_model.pt complete")

# 3) best_config.json
best_config = {
    "experiment": "E4",
    "dataset": "EMNIST Balanced",
    "seed": SEED,
    "architecture": {
        "type": "MLP",
        "input_dim": 784,
        "hidden_sizes": [512, 256],
        "activation": "ReLU",
        "dropout_p": e4_dropout,
        "use_batchnorm": e4_batchnorm,
        "num_classes": NUM_CLASSES
    },
    "training": {
        "optimizer": "Adam",
        "lr": 1e-3,
        "batch_size": BATCH_SIZE,
        "max_epochs": 30,
        "early_stopping_patience": 5,
        "epochs_trained": epochs_e4
    },
    "results": {
        "best_val_accuracy": round(best_val_e4, 4),
        "test_accuracy": round(test_acc, 4),
        "test_loss": round(test_loss, 4)
    }
}
with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config, f, indent=2, ensure_ascii=False)
print("artifacts/best_config.json complete")

print("\nВсе артефакты сохранены.")

artifacts/runs.csv complete
experiment_id         dataset  seed                 model_summary optimizer      lr  momentum  weight_decay  epochs_trained  best_val_accuracy  best_val_loss
           E1 EMNIST Balanced    42                [512,256] ReLU      Adam 0.00100       0.0        0.0000              20             0.8452         0.4814
           E2 EMNIST Balanced    42 [512,256] ReLU | Dropout(0.3)      Adam 0.00100       0.0        0.0000              20             0.8473         0.4584
           E3 EMNIST Balanced    42    [512,256] ReLU | BatchNorm      Adam 0.00100       0.0        0.0000              20             0.8517         0.4506
           E4 EMNIST Balanced    42    [512,256] ReLU | BatchNorm      Adam 0.00100       0.0        0.0000              15             0.8510         0.4506
           O1 EMNIST Balanced    42    [512,256] ReLU | BatchNorm      Adam 0.10000       0.0        0.0000               8             0.8102         0.6106
           O2 EMNIST Bal